# Build DPO Positive/Negative Pairs with Rejection Sampling

Pipeline:
1. Positive samples are loaded directly from ./data/llm_train.json.
2. For each prompt, generate N candidates with the V4.1 style model at higher temperature.
3. Score candidates with BGE semantic similarity and style classifier score.
4. Keep candidates with semantic >= threshold and valid CoT, then pick the lowest style score as rejected.
5. Export DPO pairs to JSON and JSONL.

In [1]:
# Run this on Remote Jupyter Book.

import os

os.getcwd()
os.chdir("/root/OtakuLab")

In [2]:
import json
import random
import re
from pathlib import Path
from typing import Any, Optional, TypedDict, cast

import numpy as np
import torch
import torch.nn as nn
from peft import PeftModel
from sentence_transformers import SentenceTransformer, util
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, Qwen3ForCausalLM

POSITIVE_DATA_PATH = Path("./data/llm_train.json")
DPO_OUTPUT_JSON_PATH = Path("./data/dpo_pairs_v41_rs.json")
DPO_OUTPUT_JSONL_PATH = Path("./data/dpo_pairs_v41_rs.jsonl")
LOST_SAMPLE_OUTPUT_JSON_PATH = Path("./data/dpo_pairs_v41_rs_lost_samples.json")

BASE_MODEL_PATH = Path("/root/autodl-tmp/Qwen3-1.7B/")
V41_CHECKPOINT_DIR = Path("/root/OtakuLab/outputs/styled-qwen-v4.3-4e-05")
STYLE_CLASSIFIER_DIR = Path("./outputs/evaluate/style-classifier/")
BGE_MODEL_PATH = Path("/root/autodl-tmp/bge-large-zh-v1.5")

NUM_CANDIDATES = 4
NUM_ROUNDS = 2
TEMPERATURE = 0.8
TOP_P = 0.9
MAX_NEW_TOKENS = 768
REPETITION_PENALTY = 1.1
ENABLE_THINKING = True
FORCE_INJECT_CHINESE_ANCHOR = True
SEMANTIC_THRESHOLD = 0.85
MIN_COT_CHARS = 8
GEN_BATCH_SIZE = 80

MAX_SAMPLES: Optional[int] = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Base model:", BASE_MODEL_PATH)
print("V4.1 checkpoint:", V41_CHECKPOINT_DIR)
print("Style classifier:", STYLE_CLASSIFIER_DIR)
print("BGE:", BGE_MODEL_PATH)
print("Generation batch size:", GEN_BATCH_SIZE)

/root/miniconda3/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda
Base model: /root/autodl-tmp/Qwen3-1.7B
V4.1 checkpoint: /root/OtakuLab/outputs/styled-qwen-v4.3-4e-05
Style classifier: outputs/evaluate/style-classifier
BGE: /root/autodl-tmp/bge-large-zh-v1.5
Generation batch size: 80


## Load Positive Samples and Prompt Builders

In [7]:
SYNTACTIC_DIMS = [
    "mdd",
    "verb_noun_ratio",
    "question_ratio",
    "imperative_ratio",
    "fragment_ratio",
    "marker_density",
]

EN_NAME_TO_ZH = {
    "Muice": "沐雪",
    "Ayaka": "神里绫华",
    "Zhongli": "钟离",
    "Hutao": "胡桃",
    "Haruhi": "凉宫春日",
    "Liyunlong": "李云龙",
    "Sheldon": "谢尔顿",
    "Wukong": "孙悟空",
}
ZH_NAME_TO_EN = {v: k for k, v in EN_NAME_TO_ZH.items()}

class InstructionComponents(TypedDict):
    semantic_tags: list[str]
    structural_metrics: dict[str, float]
    signature_rules: list[str]

class PositiveItem(TypedDict):
    character: str
    neutral_sentence: str
    instruction_components: InstructionComponents
    thinking_process: str
    output: str

def build_system_prompt() -> str:
    return (
        "You are an expert in stylistic rewriting. "
        "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
    )

def build_user_prompt(item: PositiveItem) -> str:
    instr = item["instruction_components"]
    semantic_tags = instr.get("semantic_tags", [])
    signature_rules = instr.get("signature_rules", [])
    semantic_text = ", ".join(semantic_tags) if semantic_tags else "None"
    # signature_text = " ".join(signature_rules) if signature_rules else "None"
    signature_text = "None"  # 剥夺词汇提示以降低模型的风格能力
    return (
        f"Target Character: {item['character']}\n"
        f"Semantic Traits: {semantic_text}\n"
        f"Formatting Signature: {signature_text}\n"
        f"Neutral Content: {item['neutral_sentence']}\n"
    )

def compose_assistant_text(cot_text: str, output_text: str) -> str:
    cot = (cot_text or "").strip().replace("<think>", "").replace("</think>", "")
    out = (output_text or "").strip()
    if cot:
        return f"<think>{cot}</think>\n\n{out}".strip()
    return out

def load_positive_items(path: Path) -> list[PositiveItem]:
    if not path.exists():
        raise FileNotFoundError(f"Missing positive data file: {path}")

    raw_data = json.loads(path.read_text(encoding="utf-8"))
    items: list[PositiveItem] = []

    for row in raw_data:
        character = str(row.get("character", "")).strip()
        neutral_sentence = str(row.get("neutral_sentence", "")).strip()
        output = str(row.get("output", "")).strip()
        thinking_process = str(row.get("thinking_process", "")).strip()
        instr_raw = row.get("instruction_components", {})

        if not character or not neutral_sentence or not output or not isinstance(instr_raw, dict):
            continue

        structural_metrics_raw = instr_raw.get("structural_metrics", {})
        if not isinstance(structural_metrics_raw, dict):
            continue

        structural_metrics = {
            dim: float(structural_metrics_raw.get(dim, 0.0))
            for dim in SYNTACTIC_DIMS
        }

        semantic_tags = instr_raw.get("semantic_tags", [])
        signature_rules = instr_raw.get("signature_rules", [])

        if not isinstance(semantic_tags, list):
            semantic_tags = []
        if not isinstance(signature_rules, list):
            signature_rules = []

        item: PositiveItem = {
            "character": character,
            "neutral_sentence": neutral_sentence,
            "instruction_components": {
                "semantic_tags": [str(x) for x in semantic_tags],
                "structural_metrics": structural_metrics,
                "signature_rules": [str(x) for x in signature_rules],
            },
            "thinking_process": thinking_process,
            "output": output,
        }
        items.append(item)

    return items

positive_items = load_positive_items(POSITIVE_DATA_PATH)
if MAX_SAMPLES is not None:
    positive_items = positive_items[:MAX_SAMPLES]

print(f"Loaded positive samples: {len(positive_items)}")

Loaded positive samples: 8272


## Load V4.1 Style Model for Candidate Generation

In [8]:
class StyleEncoder(nn.Module):
    def __init__(self, input_dim: int, out_dim: int):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, out_dim // 2),
            nn.ReLU(),
            nn.Linear(out_dim // 2, out_dim),
            nn.Tanh(),
        )

    def forward(self, syntactic_vec: torch.Tensor) -> torch.Tensor:
        return self.proj(syntactic_vec)

def ensure_generator_assets(base_model_path: Path, checkpoint_dir: Path) -> None:
    missing = []
    if not base_model_path.exists():
        missing.append(str(base_model_path))

    required_paths = [
        checkpoint_dir / "lora",
        checkpoint_dir / "style_encoder.pt",
        checkpoint_dir / "tokenizer",
    ]
    for p in required_paths:
        if not p.exists():
            missing.append(str(p))

    if missing:
        raise FileNotFoundError(
            "Missing generator assets. Please verify V4.1 paths:\n" + "\n".join(missing)
        )

ensure_generator_assets(BASE_MODEL_PATH, V41_CHECKPOINT_DIR)

gen_tokenizer = AutoTokenizer.from_pretrained(V41_CHECKPOINT_DIR / "tokenizer")
if gen_tokenizer.pad_token is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token
gen_tokenizer.padding_side = "right"

base_model = Qwen3ForCausalLM.from_pretrained(BASE_MODEL_PATH, dtype=torch.bfloat16)
gen_model = PeftModel.from_pretrained(base_model, V41_CHECKPOINT_DIR / "lora")

hidden_size_raw = base_model.config.hidden_size
if hidden_size_raw is None:
    raise ValueError("base model hidden_size is None")
hidden_size = int(hidden_size_raw)

style_encoder = StyleEncoder(len(SYNTACTIC_DIMS), hidden_size)
style_encoder.load_state_dict(torch.load(V41_CHECKPOINT_DIR / "style_encoder.pt", map_location=DEVICE))

style_encoder.to(DEVICE)
gen_model.to(DEVICE)
style_encoder.eval()
gen_model.eval()

GEN_DTYPE = next(gen_model.parameters()).dtype
print("Generator loaded. dtype =", GEN_DTYPE)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Generator loaded. dtype = torch.bfloat16


## Load Scorers: BGE Semantic + Style Classifier

In [9]:
class CharacterStyleClassifier(nn.Module):
    def __init__(self, backbone_name: str, embed_dim: int, proj_dim: int, num_roles: int, dropout: float = 0.4):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_name, output_hidden_states=False)
        self.hidden_size = self.backbone.config.hidden_size
        if self.hidden_size != embed_dim:
            raise ValueError(f"hidden_size mismatch: {self.hidden_size} != {embed_dim}")

        self.proj = nn.Sequential(
            nn.Linear(self.hidden_size, proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(proj_dim, num_roles)

    @staticmethod
    def mean_pooling(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_hidden = torch.sum(last_hidden_state * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        return sum_hidden / sum_mask

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        sent_emb = self.mean_pooling(out.last_hidden_state, attention_mask)
        proj = self.proj(sent_emb)
        logits = self.classifier(proj)
        return logits, proj

def load_style_classifier(path: Path, device: torch.device):
    if not path.exists():
        raise FileNotFoundError(f"Style classifier dir not found: {path}")

    with open(path / "label_encoder.json", "r", encoding="utf-8") as f:
        le_json = json.load(f)

    label_encoder = LabelEncoder()
    label_encoder.classes_ = np.array(le_json["classes"])

    hidden_size = AutoModel.from_pretrained(path).config.hidden_size
    model = CharacterStyleClassifier(
        backbone_name=str(path),
        embed_dim=hidden_size,
        proj_dim=256,
        num_roles=len(label_encoder.classes_),
        dropout=0.4,
    )

    chk = torch.load(path / "head.pt", map_location=device)
    model.proj.load_state_dict(chk["proj_state"])
    model.classifier.load_state_dict(chk["classifier_state"])
    model.to(device).eval()

    tokenizer = AutoTokenizer.from_pretrained(path)

    centers_npz = np.load(path / "role_centers.npz")
    role_centers = {k: centers_npz[k] for k in centers_npz.files}

    return model, tokenizer, label_encoder, role_centers

def resolve_center_key(character: str, role_centers: dict[str, np.ndarray]) -> Optional[str]:
    candidates = [character]
    zh = EN_NAME_TO_ZH.get(character)
    en = ZH_NAME_TO_EN.get(character)
    if zh:
        candidates.append(zh)
    if en:
        candidates.append(en)

    for key in candidates:
        if key in role_centers:
            return key
    return None

def get_style_score(
    model: nn.Module,
    tokenizer,
    text: str,
    role_center: np.ndarray,
    device: torch.device,
    max_length: int = 128,
) -> float:
    model.eval()
    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        padding="max_length",
        return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        _, emb = model(input_ids=input_ids, attention_mask=attention_mask)

    emb_np = emb.detach().cpu().numpy()[0]
    num = float(np.dot(emb_np, role_center))
    den = float(np.linalg.norm(emb_np) * np.linalg.norm(role_center) + 1e-9)
    return num / den

def compute_semantic_similarity(semantic_model: SentenceTransformer, neutral_text: str, stylized_text: str) -> float:
    if not neutral_text.strip() or not stylized_text.strip():
        return 0.0

    embeddings = semantic_model.encode(
        [neutral_text, stylized_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return float(util.cos_sim(embeddings[0], embeddings[1]).item())

style_model, style_tokenizer, style_label_encoder, role_centers = load_style_classifier(
    STYLE_CLASSIFIER_DIR,
    DEVICE,
)
semantic_model = SentenceTransformer(str(BGE_MODEL_PATH), device=str(DEVICE))

print("Loaded style classifier labels:", style_label_encoder.classes_)
print("Loaded role centers:", len(role_centers))
print("Semantic scorer ready.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /root/autodl-tmp/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded style classifier labels: ['凉宫春日' '孙悟空' '李云龙' '沐雪' '神里绫华' '胡桃' '谢尔顿' '钟离']
Loaded role centers: 8
Semantic scorer ready.


## Candidate Generation, Parsing, and Rejection Sampling

In [ ]:
# pyright: reportCallIssue=false

def clean_candidate_text(text: str) -> tuple[str, int]:
    raw = (text or "").strip()
    if not raw:
        return "", 0

    kept: list[str] = []
    for line in raw.splitlines():
        s = line.strip()
        if not s:
            continue

        low = s.lower()
        if low.startswith("target character:"):
            continue
        if low.startswith("semantic traits:"):
            continue
        if low.startswith("formatting signature:"):
            continue
        if low.startswith("neutral content:"):
            continue

        kept.append(s)

    cleaned = "\n".join(kept).strip()
    cleaned = re.sub(
        r"^(assistant|response|answer|回答)\s*[:：]\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    ).strip()

    removed_chars = max(0, len(raw) - len(cleaned))
    return cleaned, removed_chars


def strip_prompt_prefix(decoded_text: str, prompt_text: str) -> str:
    text = decoded_text.strip()
    prompt = prompt_text.strip()
    if text.startswith(prompt):
        return text[len(prompt):].strip()

    for window in (256, 128, 64, 32):
        if len(prompt) >= window:
            tail = prompt[-window:]
            if text.startswith(tail):
                return text[window:].strip()
    return text


def extract_cot_and_output(generated_text: str) -> tuple[str, str, dict[str, Any]]:
    text = generated_text.strip()

    think_match = re.search(r"<think>(.*?)</think>", text, flags=re.IGNORECASE | re.DOTALL)
    has_think_tag = think_match is not None

    if think_match:
        raw_cot = think_match.group(1).strip()
        raw_output = text[think_match.end():].strip()
    else:
        raw_cot = ""
        raw_output = text

    cot_text, cot_removed_chars = clean_candidate_text(raw_cot)
    output_text, output_removed_chars = clean_candidate_text(raw_output)

    # 保守策略：若 CoT 被清洗为空，回退到原始 CoT，避免过度清洗丢失推理文本。
    if raw_cot and not cot_text:
        cot_text = raw_cot

    diagnostics = {
        "has_think_tag": has_think_tag,
        "raw_cot": raw_cot,
        "raw_output": raw_output,
        "cot_removed_chars": cot_removed_chars,
        "output_removed_chars": output_removed_chars,
        "cleaning_removed_chars": cot_removed_chars + output_removed_chars,
    }
    return cot_text, output_text, diagnostics


@torch.inference_mode()
def generate_candidates_batch(items: list[PositiveItem], temperature: float) -> list[dict[str, Any]]:
    if not items:
        return []

    system_prompt = build_system_prompt()
    prompt_texts: list[str] = []
    structural_rows: list[list[float]] = []

    for item in items:
        user_prompt = build_user_prompt(item)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        prompt_text = gen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )

        if ENABLE_THINKING and FORCE_INJECT_CHINESE_ANCHOR:
            prompt_text = prompt_text.rstrip() + " " + "<think>思考: "

        prompt_texts.append(prompt_text)

        structural_metrics = item["instruction_components"].get("structural_metrics", {})
        structural_rows.append([float(structural_metrics.get(dim, 0.0)) for dim in SYNTACTIC_DIMS])

    # 批量生成时使用左侧填充，保证因果注意力对齐。
    original_padding_side = gen_tokenizer.padding_side
    gen_tokenizer.padding_side = "left"
    try:
        tokenized = gen_tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=768,
        ).to(DEVICE)
    finally:
        gen_tokenizer.padding_side = original_padding_side

    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    syntactic_tensor = torch.tensor(
        structural_rows,
        dtype=torch.float32,
        device=DEVICE,
    )

    style_dtype = next(style_encoder.parameters()).dtype
    style_emb = style_encoder(syntactic_tensor.to(style_dtype)).to(GEN_DTYPE)
    style_prefix = style_emb.unsqueeze(1)

    embedding_layer = cast(nn.Module, gen_model.get_input_embeddings())
    token_embeds = embedding_layer(input_ids)
    inputs_embeds = torch.cat([style_prefix, token_embeds], dim=1)

    prefix_mask = torch.ones((attention_mask.shape[0], 1), dtype=torch.long, device=DEVICE)
    new_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

    generated_ids = gen_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=new_attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=temperature,
        top_p=TOP_P,
        do_sample=True,
        repetition_penalty=REPETITION_PENALTY,
        pad_token_id=gen_tokenizer.pad_token_id,
    )

    decoded_batch = gen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    generated_rows: list[dict[str, Any]] = []
    for decoded, prompt_text in zip(decoded_batch, prompt_texts):
        decoded = "<think>" + decoded if ENABLE_THINKING and FORCE_INJECT_CHINESE_ANCHOR else decoded
        generated_text = strip_prompt_prefix(decoded, prompt_text)
        cot_text, output_text, parse_diag = extract_cot_and_output(generated_text)

        generated_rows.append(
            {
                "prompt": prompt_text,
                "decoded": decoded,
                "generated_text": generated_text,
                "cot": cot_text,
                "output": output_text,
                **parse_diag,
            }
        )

    return generated_rows


def compute_semantic_similarity_batch(
    semantic_model: SentenceTransformer,
    neutral_texts: list[str],
    stylized_texts: list[str],
) -> list[float]:
    if len(neutral_texts) != len(stylized_texts):
        raise ValueError("neutral_texts and stylized_texts size mismatch")

    scores = [0.0] * len(neutral_texts)
    valid_indices = [
        i
        for i, (neutral_text, stylized_text) in enumerate(zip(neutral_texts, stylized_texts))
        if neutral_text.strip() and stylized_text.strip()
    ]
    if not valid_indices:
        return scores

    valid_neutral = [neutral_texts[i] for i in valid_indices]
    valid_stylized = [stylized_texts[i] for i in valid_indices]

    embeddings = semantic_model.encode(
        valid_neutral + valid_stylized,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    split_at = len(valid_indices)
    neutral_emb = embeddings[:split_at]
    stylized_emb = embeddings[split_at:]
    semantic_values = np.sum(neutral_emb * stylized_emb, axis=1)

    for pos, idx in enumerate(valid_indices):
        scores[idx] = float(semantic_values[pos])

    return scores


def get_style_score_batch(
    model: nn.Module,
    tokenizer,
    texts: list[str],
    role_centers_batch: list[np.ndarray],
    device: torch.device,
    max_length: int = 128,
) -> list[float]:
    if len(texts) != len(role_centers_batch):
        raise ValueError("texts and role_centers_batch size mismatch")

    scores = [-1.0] * len(texts)
    valid_indices = [i for i, text in enumerate(texts) if text.strip()]
    if not valid_indices:
        return scores

    valid_texts = [texts[i] for i in valid_indices]
    enc = tokenizer(
        valid_texts,
        truncation=True,
        max_length=max_length,
        padding="max_length",
        return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    model.eval()
    with torch.no_grad():
        _, emb = model(input_ids=input_ids, attention_mask=attention_mask)

    emb_np = emb.detach().cpu().numpy()
    center_np = np.stack([role_centers_batch[i] for i in valid_indices], axis=0)

    num = np.sum(emb_np * center_np, axis=1)
    den = np.linalg.norm(emb_np, axis=1) * np.linalg.norm(center_np, axis=1) + 1e-9
    batch_scores = num / den

    for pos, idx in enumerate(valid_indices):
        scores[idx] = float(batch_scores[pos])

    return scores


def summarize_candidate_pool(candidate_pool: list[dict[str, Any]]) -> dict[str, int]:
    return {
        "total_candidates": len(candidate_pool),
        "non_empty_output": sum(1 for c in candidate_pool if c.get("valid_output", False)),
        "semantic_pass": sum(1 for c in candidate_pool if c.get("valid_semantic", False)),
        "cot_pass": sum(1 for c in candidate_pool if c.get("valid_cot", False)),
        "eligible": sum(1 for c in candidate_pool if c.get("eligible", False)),
        "with_think_tag": sum(1 for c in candidate_pool if c.get("has_think_tag", False)),
        "cleaned_to_empty": sum(1 for c in candidate_pool if c.get("cleaned_to_empty", False)),
    }


def build_lost_record(
    sample_idx: int,
    item: PositiveItem,
    status: str,
    candidate_pool: list[dict[str, Any]],
) -> dict[str, Any]:
    reason_map = {
        "missing_role_center": "style classifier role center missing for character",
        "no_eligible_candidate": "no candidate satisfies non-empty output + semantic threshold + minimum CoT length",
    }
    return {
        "sample_idx": sample_idx,
        "character": item["character"],
        "neutral_sentence": item["neutral_sentence"],
        "status": status,
        "lost_reason": reason_map.get(status, status),
        "candidate_summary": summarize_candidate_pool(candidate_pool),
        "candidate_pool": candidate_pool,
    }


def select_negative_candidate_from_pool(candidate_pool: list[dict[str, Any]]) -> tuple[Optional[dict[str, Any]], str]:
    # 优先筛选出语义分达标（例如 Semantic $> 0.85$）但风格分最低的那个样本
    for round_idx in range(1, NUM_ROUNDS + 1):
        eligible_candidates = [
            c for c in candidate_pool if c.get("round", 0) <= round_idx and c.get("eligible", False)
        ]
        if eligible_candidates:
            negative = min(eligible_candidates, key=lambda x: x["style_score"])
            return negative, "ok"

    # 如果全部样本的语义分都小于阈值，就选语义分最高的那个，虽然它不满足语义阈值，但至少是相对最好的。
    negative = max(candidate_pool, key=lambda x: x.get("semantic_score", -1.0), default=None)
    if negative and negative.get("semantic_score", 0.0) > 0.0:
        return negative, "ok"

    return None, "no_eligible_candidate"


class CandidateJob(TypedDict):
    sample_idx: int
    round: int
    candidate_idx: int


def build_dpo_pairs(
    items: list[PositiveItem],
    generation_batch_size: int = GEN_BATCH_SIZE,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, int]]:
    generation_batch_size = max(1, int(generation_batch_size))

    stats: dict[str, int] = {
        "total": len(items),
        "paired": 0,
        "lost_total": 0,
        "missing_role_center": 0,
        "no_eligible_candidate": 0,
    }

    pairs: list[dict[str, Any]] = []
    lost_samples: list[dict[str, Any]] = []

    candidate_pools: dict[int, list[dict[str, Any]]] = {
        sample_idx: [] for sample_idx in range(1, len(items) + 1)
    }
    sample_to_role_center: dict[int, np.ndarray] = {}
    missing_role_center_indices: set[int] = set()

    jobs: list[CandidateJob] = []

    for sample_idx, item in enumerate(items, start=1):
        center_key = resolve_center_key(item["character"], role_centers)
        if center_key is None:
            missing_role_center_indices.add(sample_idx)
            continue

        sample_to_role_center[sample_idx] = role_centers[center_key]

        for round_idx in range(NUM_ROUNDS):
            for candidate_idx in range(NUM_CANDIDATES):
                jobs.append(
                    {
                        "sample_idx": sample_idx,
                        "round": round_idx + 1,
                        "candidate_idx": candidate_idx + 1,
                    }
                )

    for start in tqdm(
        range(0, len(jobs), generation_batch_size),
        desc=f"Generating candidates (bs={generation_batch_size})",
    ):
        batch_jobs = jobs[start : start + generation_batch_size]
        batch_items = [items[job["sample_idx"] - 1] for job in batch_jobs]

        generated_rows = generate_candidates_batch(batch_items, temperature=TEMPERATURE)
        valid_positions = [i for i, row in enumerate(generated_rows) if row is not None]
        if not valid_positions:
            continue

        valid_neutral_texts = [batch_items[i]["neutral_sentence"] for i in valid_positions]
        valid_output_texts = [
            cast(dict[str, Any], generated_rows[i]).get("output", "").strip() for i in valid_positions
        ]
        valid_cot_texts = [
            cast(dict[str, Any], generated_rows[i]).get("cot", "").strip() for i in valid_positions
        ]
        valid_role_centers = [
            sample_to_role_center[batch_jobs[i]["sample_idx"]] for i in valid_positions
        ]

        semantic_scores = compute_semantic_similarity_batch(
            semantic_model,
            valid_neutral_texts,
            valid_output_texts,
        )
        style_scores = get_style_score_batch(
            style_model,
            style_tokenizer,
            valid_output_texts,
            valid_role_centers,
            device=DEVICE,
        )

        for pos, row_idx in enumerate(valid_positions):
            generated = cast(dict[str, Any], generated_rows[row_idx])
            output_text = valid_output_texts[pos]
            cot_text = valid_cot_texts[pos]

            valid_output = bool(output_text)
            semantic_score = float(semantic_scores[pos]) if valid_output else 0.0
            style_score = float(style_scores[pos]) if valid_output else -1.0
            valid_semantic = bool(semantic_score >= SEMANTIC_THRESHOLD)
            valid_cot = bool(len(cot_text) >= MIN_COT_CHARS)
            eligible = valid_output and valid_semantic and valid_cot

            job = batch_jobs[row_idx]
            candidate = {
                "round": job["round"],
                "candidate_idx": job["candidate_idx"],
                "cot": cot_text,
                "output": output_text,
                "semantic_score": semantic_score,
                "style_score": style_score,
                "valid_output": valid_output,
                "valid_semantic": valid_semantic,
                "valid_cot": valid_cot,
                "eligible": eligible,
                "has_think_tag": bool(generated.get("has_think_tag", False)),
                "cleaning_removed_chars": int(generated.get("cleaning_removed_chars", 0)),
                "cleaned_to_empty": bool((not valid_output) and generated.get("raw_output", "").strip()),
            }
            candidate_pools[job["sample_idx"]].append(candidate)

    for sample_idx, item in enumerate(tqdm(items, desc="Selecting rejected pairs"), start=1):
        candidate_pool = candidate_pools.get(sample_idx, [])

        if sample_idx in missing_role_center_indices:
            status = "missing_role_center"
            stats[status] = stats.get(status, 0) + 1
            stats["lost_total"] += 1
            lost_samples.append(build_lost_record(sample_idx, item, status, candidate_pool))
            continue

        negative, status = select_negative_candidate_from_pool(candidate_pool)
        if status != "ok" or negative is None:
            stats[status] = stats.get(status, 0) + 1
            stats["lost_total"] += 1
            lost_samples.append(build_lost_record(sample_idx, item, status, candidate_pool))
            continue

        chosen_cot = item.get("thinking_process", "")
        chosen_output = item.get("output", "")
        rejected_cot = negative.get("cot", "")
        rejected_output = negative.get("output", "")

        pair = {
            "prompt": [
                {"role": "system", "content": build_system_prompt()},
                {"role": "user", "content": build_user_prompt(item)},
            ],
            "chosen": compose_assistant_text(chosen_cot, chosen_output),
            "rejected": compose_assistant_text(rejected_cot, rejected_output),
            "meta": {
                "character": item["character"],
                "negative_semantic_score": float(negative["semantic_score"]),
                "negative_style_score": float(negative["style_score"]),
            },
        }

        pairs.append(pair)
        stats["paired"] += 1

    return pairs, lost_samples, stats

## Build and Export DPO Dataset

> Lost 样本构建过程（透明记录）
1. 对每个正样本生成 $N \times R$ 个候选。
2. 每个候选按 `valid_output`、`semantic>=threshold`、`cot_length>=MIN_COT_CHARS` 三条件判定是否可用。
3. 若可用候选非空，选风格分最低者作为 rejected；否则该样本记为 lost。
4. lost 样本会导出原始状态、候选统计与候选池，便于定位“语义不过线 / CoT 过短 / 输出被清洗为空”等失败原因。

In [11]:
pairs, lost_samples, stats = build_dpo_pairs(positive_items, generation_batch_size=GEN_BATCH_SIZE)

DPO_OUTPUT_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
DPO_OUTPUT_JSON_PATH.write_text(
    json.dumps(pairs, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

with DPO_OUTPUT_JSONL_PATH.open("w", encoding="utf-8") as f:
    for row in pairs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

LOST_SAMPLE_OUTPUT_JSON_PATH.write_text(
    json.dumps(lost_samples, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("DPO build done.")
print("Stats:", stats)
if stats.get("total", 0) > 0:
    pair_ratio = stats.get("paired", 0) / stats["total"]
    lost_ratio = stats.get("lost_total", 0) / stats["total"]
    print(f"Pair ratio: {pair_ratio:.2%}")
    print(f"Lost ratio: {lost_ratio:.2%}")
print("Saved JSON :", DPO_OUTPUT_JSON_PATH)
print("Saved JSONL:", DPO_OUTPUT_JSONL_PATH)
print("Saved lost JSON :", LOST_SAMPLE_OUTPUT_JSON_PATH)

Generating candidates (bs=80):   0%|          | 0/828 [00:00<?, ?it/s]

Selecting rejected pairs: 100%|██████████| 8272/8272 [00:00<00:00, 83178.58it/s]


DPO build done.
Stats: {'total': 8272, 'paired': 8272, 'lost_total': 0, 'missing_role_center': 0, 'no_eligible_candidate': 0}
Pair ratio: 100.00%
Lost ratio: 0.00%
Saved JSON : data/dpo_pairs_v41_rs.json
Saved JSONL: data/dpo_pairs_v41_rs.jsonl
Saved lost JSON : data/dpo_pairs_v41_rs_lost_samples.json


In [12]:
print("=== Pair Preview ===")
preview_n = min(2, len(pairs))
for i in range(preview_n):
    print(f"--- Pair {i + 1} ---")
    print("Character:", pairs[i]["meta"]["character"])
    print(
        "Negative semantic/style:",
        pairs[i]["meta"]["negative_semantic_score"],
        pairs[i]["meta"]["negative_style_score"],
    )
    print("Chosen output:", pairs[i]["chosen"])
    print("Rejected output:", pairs[i]["rejected"])
    print()

print("=== Lost Sample Construction Summary ===")
print("Rule: lost = no eligible candidate after rejection sampling OR missing role center")
print("Eligibility = valid_output AND semantic>=threshold AND cot_length>=MIN_COT_CHARS")
print("Lost count:", len(lost_samples))

lost_preview_n = min(2, len(lost_samples))
for i in range(lost_preview_n):
    row = lost_samples[i]
    print(f"--- Lost {i + 1} ---")
    print("Character:", row["character"])
    print("Status:", row["status"])
    print("Reason:", row["lost_reason"])
    print("Candidate summary:", row["candidate_summary"])
    print()

# 语义分 < 0.85 的样本
low_semantic_rejected_outputs = []
for pair in pairs:
    if pair["meta"]["negative_semantic_score"] < SEMANTIC_THRESHOLD:
        low_semantic_rejected_outputs.append(
            {
                "character": pair["meta"]["character"],
                "negative_semantic_score": pair["meta"]["negative_semantic_score"],
                "negative_style_score": pair["meta"]["negative_style_score"],
                "rejected_output": pair["rejected"],
            }
        )
print("=== Low Semantic Score Rejected Outputs (Semantic < 0.85) ===")
print("count:", len(low_semantic_rejected_outputs))
for i, item in enumerate(low_semantic_rejected_outputs[:3]):
    print(f"--- Low Semantic {i + 1} ---")
    print("Character:", item["character"])
    print("Negative semantic score:", item["negative_semantic_score"])
    print("Negative style score:", item["negative_style_score"])
    print("Rejected output:", item["rejected_output"])
    print()


=== Pair Preview ===
--- Pair 1 ---
Character: 沐雪
Negative semantic/style: 0.9588634967803955 0.5115474462509155
Chosen output: <think>她依天真语义选择发问，极高碎片率与疑问率使句式松散，最后用“这个嘛”及省略号、括号动作展现呆萌可爱的身份特征。</think>

这个嘛...奇奇怪怪的东西指的是什么呢（天真）
Rejected output: <think>作为天真好奇的角色，她用直率反问引发互动；高疑问比与碎片化节奏制造轻快跳跃感，虽无特定符号但通过活泼语气传递可爱。</think>

“奇怪的东西”指的是什么？

--- Pair 2 ---
Character: 沐雪
Negative semantic/style: 0.883599042892456 0.626469075679779
Chosen output: <think>她依活力与可爱语义选用夸张比喻，极高碎片率与祈使感加快节奏，最后用“根本”“啦”配合多重感叹号，展现元气与调皮。</think>

不用担心哦，我的身体可是用钛合金打造的，根本不会受伤哦！与其担心我的身体，不如担心你的又小又弱的身体啦！
Rejected output: <think>作为热情且乐观的个性，我以“元气”和“可爱”的语义强化语气；零标记密度与高碎片率将长句拆解为轻快短句；最后通过省略号、重复感叹号及口语化表达体现活泼可爱的签名。</think>

不用担心啦！我的身体可不会受伤哦！你该好好照顾自己才是。

=== Lost Sample Construction Summary ===
Rule: lost = no eligible candidate after rejection sampling OR missing role center
Eligibility = valid_output AND semantic>=threshold AND cot_length>=MIN_COT_CHARS
Lost count: 0
=== Low Semantic Score Rejected Outputs (Semantic < 0.85) ===
count: 252
---